# Monster Generator: teach diffusion by making tiny creatures
**A from-scratch, text-conditioned diffusion lab for high-school students.**

Enter `happy green monster with three eyes and horns`, then watch noise turn into a monster.

- Original **32 × 32 RGB** artwork; **13,200 training images** already on GitHub.
- Learnable **32-number word embeddings**; a small conditional U-Net predicts noise.
- No model account, API key, or paid AI service. A Colab GPU is required for the full training lesson.
- **A100 target:** approximately 30 minutes for setup, training and first samples; this is a target, not a measured A100 guarantee. Training stops after 24 minutes and saves its progress. T4 can take longer or need more training.

**Before running:** Runtime → Change runtime type → Python 3 → GPU (A100 if available). Then Runtime → Run all. The creative controls appear near the end. The first run trains the model; later runs can resume the saved checkpoint.

### Learning path
1. Inspect monsters and their captions.
2. Add known noise to a clean image.
3. Teach a neural network to predict that noise.
4. Start with fresh noise and denoise repeatedly.
5. Change a word, change a seed, and test combinations the model never saw.

The notebook and explanations are in English for classroom use. See `TEACHER_GUIDE.md` for a lesson plan and discussion prompts.

## 1. Set up and download the prepared dataset
The small source files and the ~2 MB dataset are downloaded from the teaching repository. Each download is checked against a SHA-256 checksum. **The monster renderer does not run during this notebook.**

In [ ]:
import os, sys, subprocess, importlib.util, hashlib, urllib.request
from pathlib import Path

SMOKE = os.environ.get('MONSTER_SMOKE') == '1'  # Used only by the automated CPU smoke check.
packages = {'numpy': 'numpy>=1.26,<3', 'PIL': 'Pillow>=10,<13',
            'matplotlib': 'matplotlib>=3.8,<4', 'ipywidgets': 'ipywidgets>=8,<9'}
missing = [requirement for module, requirement in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
import torch
if tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2]) < (2, 3):
    raise RuntimeError('Use a current Colab runtime with PyTorch 2.3 or newer.')

PROJECT = Path(os.environ.get('MONSTER_LOCAL_DIR', '/content/monster_diffusion'))
PROJECT.mkdir(parents=True, exist_ok=True)
RAW = 'https://raw.githubusercontent.com/dujing82-blip/tiny-text-to-pixel-ai/main/monster_diffusion'
EXPECTED_HASHES = {'dataset.py': '3a781ac064fd3dc70df9410c5967c230ab89a8548b0f1b78fde7827764c2cc10', 'model.py': 'b6c9d1a64afbffd6d2a89c42a39043917056bb237d6800adac7ee8758fa1a765', 'training.py': 'aeb1579032c4c1002da37e643e95db4b8c970776c5af258eb907a98ceb05cb72', 'visuals.py': '4b299aae1ff76c36bb08a1cca4648b5e210ac386a2de8ea35eda2d60c6eb424d', 'data/monsters32.npz': '103c7e0ac28174378fd97f13e413e4a03905b75c3d70e746ea25d99ef3e49a11', 'data/metadata.json': 'b0a1657014de9f752aeb6184fb622e0be7673173afe9d8da0720f2731d44d324'}

def verified_download(name):
    destination = PROJECT / name
    expected = EXPECTED_HASHES[name]
    if destination.exists() and hashlib.sha256(destination.read_bytes()).hexdigest() == expected:
        return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(3):
        try:
            with urllib.request.urlopen(f'{RAW}/{name}', timeout=40) as response:
                payload = response.read()
            if hashlib.sha256(payload).hexdigest() != expected:
                raise ValueError(f'Checksum mismatch: {name}. Reopen the current GitHub notebook.')
            destination.write_bytes(payload)
            return destination
        except Exception:
            if attempt == 2:
                raise

if not SMOKE:
    for name in EXPECTED_HASHES:
        verified_download(name)
    if not torch.cuda.is_available():
        raise RuntimeError('GPU not enabled. Choose Runtime > Change runtime type > GPU, then rerun.')
sys.path.insert(0, str(PROJECT))
import json, numpy as np, matplotlib.pyplot as plt
from IPython.display import display, Image as DisplayImage, Markdown
from dataset import caption, parse_prompt, HELD_OUT, COLORS, EYES, HORNS, MOODS
from model import MonsterUNet, Diffusion
from training import Trainer
from visuals import show_grid, save_gif, to_rgb
if SMOKE:
    torch.set_num_threads(2)
else:
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('high')
print('PyTorch:', torch.__version__)
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU smoke check')

## 2. Inspect real training examples
Here “real” means **actual examples in our synthetic dataset**, not model-generated samples.

| Attribute | Choices |
|---|---|
| Color | green, blue, red, purple |
| Eye count | one, two, three |
| Horns | horns, no horns |
| Expression | happy, sad |

There are **48 possible combinations**. We train on 44 combinations and reserve four complete combinations for a generalization experiment. Another validation set uses the same 44 combinations with fresh rendering variations.

In [ ]:
DATA_PATH = PROJECT / 'data/monsters32.npz'
with np.load(DATA_PATH, allow_pickle=False) as archive:
    images = archive['images'].copy()
    labels = archive['labels'].copy()
    splits = archive['splits'].copy()
metadata = json.loads((PROJECT / 'data/metadata.json').read_text())
print('Images:', images.shape, images.dtype)
print('Split counts:', metadata['counts'])
rng = np.random.default_rng(42)
example_ids = rng.choice(np.flatnonzero(splits == 0), 12, replace=False)
show_grid(images[example_ids], [caption(a) for a in labels[example_ids]],
          heading='Dataset examples — original monster artwork')

## 3. Forward diffusion: add a little more noise
A diffusion model learns to undo a process we already know how to do: **adding noise**.

\[
x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\epsilon,
\qquad \epsilon\sim\mathcal N(0,I).
\]

- $x_0$: the original image, scaled from pixel values 0–255 to **−1 to 1**.
- $\epsilon$: random Gaussian noise, sampled independently for each pixel/channel.
- $t$: the noise level. Here $t=0$ is clean, and $t=256$ is almost pure noise.
- $\alpha_t=1-\beta_t$: the fraction of signal retained by one noising step.
- $\bar\alpha_t=\prod_{s=1}^t\alpha_s$: cumulative signal retained. The bar indicates the product over earlier steps.
- The square roots convert variance fractions to amplitude multipliers. They do **not** sum to one.

We can jump directly to **any noise level** with this formula. We do not have to run all the earlier steps. The implementation stores the 256 noisy levels at array indices 0–255; the clean image is displayed separately.

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
noise_process = Diffusion(timesteps=256, device=DEVICE)
x0 = torch.from_numpy(images[example_ids[:1]].copy()).permute(0, 3, 1, 2).float().to(DEVICE)/127.5-1
noise_rng = torch.Generator(device=DEVICE).manual_seed(123)
epsilon = torch.randn(x0.shape, device=DEVICE, generator=noise_rng)
levels = [1, 32, 96, 160, 224, 256]
noised = [x0.cpu()]
noise_titles = ['t=0\nclean image']
for level in levels:
    t = torch.tensor([level-1], device=DEVICE)
    noised.append(noise_process.add_noise(x0, t, epsilon).cpu())
    a = noise_process.alpha_bar[level-1].item()
    noise_titles.append(f't={level}\nalpha_bar={a:.4f}')
show_grid(torch.cat(noised), noise_titles, columns=7, heading='Forward diffusion: image → noise')
print('The same epsilon is reused here to make the change easy to see.')
print('Display clips values outside [-1, 1]; training uses the un-clipped noisy tensor.')

## 4. What does the prompt become inside the network?
`happy green monster with three eyes and horns` becomes four categorical word IDs:
**green / three / horns / happy**.

Each ID selects a **trainable vector of 32 numbers**. Concatenating four vectors gives 128 numbers; a small network turns them into the text condition. A time embedding tells the U-Net how noisy the image currently is. These two conditions enter every residual block.

**This is a controlled-language experiment.** It learns embeddings from scratch and supports the listed attributes, not arbitrary English. `smiling` means `happy`, `frowning` means `sad`, and eye counts can be written as 1/2/3. Omitted expression defaults to happy; omitted horns defaults to no horns. Unsupported words produce a clear error.

The U-Net compresses spatial resolution **32 → 16 → 8**, then expands it back to 32. Skip connections preserve local detail. Its output has the same shape as the image and estimates the added noise $\hat\epsilon$.

In [ ]:
prompt_example = 'happy green monster with three eyes and horns'
example_labels = parse_prompt(prompt_example)
print('Prompt:', prompt_example)
print('IDs [color, eyes, horns, mood]:', example_labels.tolist())
print('Canonical interpretation:', caption(example_labels))

## 5. Configure the classroom training run
The model sees a clean image and its caption, samples a random $t$ and known noise $\epsilon$, and predicts the noise:

\[
\mathcal L=\operatorname{mean}\left[(\epsilon-\epsilon_\theta(x_t,t,c))^2\right].
\]

**Why is no human labeling needed for the noise?** We created it ourselves.

Start with **4,000 steps, batch 128, 24-minute training budget**. The training log measures your actual steps/second and estimates the remaining time. The budget limits this training call; setup, saving, and sampling take additional time. A step or a preview can finish just after the budget.

Set **RESUME=True** after an interrupted run. The checkpoint includes model weights, EMA weights, optimizer, random stream and training history. `TRAIN_STEPS` always means **additional** steps in this call. Rerunning with RESUME=False starts a fresh model.

The sampling model uses an exponential moving average (EMA) of training weights, which usually makes its output steadier. Ten percent of training examples temporarily hide their text condition so the model can also learn unconditional noise prediction.

In [ ]:
TRAIN_STEPS = 4000  #@param {type:"integer"}
BATCH_SIZE = 128  #@param {type:"integer"}
TRAIN_MINUTES = 24  #@param {type:"number"}
RESUME = False  #@param {type:"boolean"}
SEED = 42  #@param {type:"integer"}
RUNS = PROJECT / 'runs'
RUNS.mkdir(exist_ok=True)
CHECKPOINT = RUNS / 'monster.pt'
if RESUME:
    if not CHECKPOINT.exists():
        raise FileNotFoundError('No checkpoint here. Restore monster.pt or set RESUME=False.')
    trainer = Trainer.resume(CHECKPOINT, DATA_PATH, device=DEVICE)
    print('Restored step', trainer.step, '; using saved batch size', trainer.config['batch_size'])
else:
    trainer = Trainer(DATA_PATH, device=DEVICE, base=8 if SMOKE else 32,
                      batch_size=4 if SMOKE else BATCH_SIZE, seed=SEED)
print('Trainable parameters:', f'{sum(p.numel() for p in trainer.model.parameters()):,}')
print('Actual training batch:', trainer.config['batch_size'])

## 6. Train the noise predictor
Watch the noise MSE decrease from around 1. **A lower noise loss does not prove that the prompt was followed**: judge color, eye count, horns and expression in the generated samples too.

A small sample preview appears every 1,000 steps. These are neural-network outputs, not drawings from the data renderer. At early steps, malformed monsters are expected.

In [ ]:
preview_prompts = [
    'happy blue monster with two eyes and horns',
    'sad green monster with one eye and no horns',
    'happy red monster with three eyes and no horns',
    'sad purple monster with two eyes and horns',
]
def training_preview(current, record):
    if current.step % 1000 == 0 and not SMOKE:
        samples, _ = current.diffusion.sample(current.ema,
            [parse_prompt(p) for p in preview_prompts], steps=48, guidance=2, seed=111)
        show_grid(samples, preview_prompts, heading=f'Learned samples after {current.step:,} steps')

trainer.train(additional_steps=2 if SMOKE else TRAIN_STEPS,
              max_minutes=1 if SMOKE else TRAIN_MINUTES,
              checkpoint=CHECKPOINT, log_every=2 if SMOKE else 100,
              callback=training_preview)
if trainer.step < 1000:
    print('Very short training run: continue training before judging image quality.')

## 7. Inspect training progress and validation
Validation images never enter the gradient update. We use fixed validation noise so repeated evaluations are comparable.

The held-out-combination MSE below is a **noise-prediction diagnostic**, not a score for visual correctness or proof of compositional generalization.

In [ ]:
history = trainer.history
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot([r['step'] for r in history], [r['train_noise_mse'] for r in history], label='training noise MSE')
validation_records = [r for r in history if 'validation_noise_mse' in r]
if validation_records:
    ax.plot([r['step'] for r in validation_records],
            [r['validation_noise_mse'] for r in validation_records], 'o-', label='validation noise MSE')
ax.set(xlabel='Optimization step', ylabel='Noise prediction MSE', title='Learning to predict the added noise')
ax.legend(); ax.grid(alpha=0.2); fig.tight_layout(); plt.show()
print('Seen-combination validation MSE:', round(trainer.validate(1, count=8 if SMOKE else 128), 4))
print('Unseen-combination diagnostic MSE:', round(trainer.validate(2, count=8 if SMOKE else 128), 4))

## 8. Make your own monsters
Change the prompt, seed or guidance below and rerun this cell.

- **Seed:** selects the initial random noise. Same model + prompt + seed + sampling settings gives repeatable output on the same device/software. A new seed gives new variation.
- **Sampling steps:** 64 by default. Training used 256 noise levels; DDIM can visit fewer levels during generation. Try 16 / 32 / 64 / 128.
- **Guidance:** 1 uses the conditional prediction directly. Larger values emphasize how the conditional prediction differs from the unconditional one; very large values can distort an image.

\[
\hat\epsilon_{\mathrm{guided}}=\hat\epsilon_{\mathrm{uncond}}+
 w(\hat\epsilon_{\mathrm{cond}}-\hat\epsilon_{\mathrm{uncond}}).
\]

The sampler below starts with pure noise and uses the **learned network at every step**. It does not call the monster renderer. More training can help when eyes or mouth shapes are unclear, but correct attributes are not guaranteed.

In [ ]:
PROMPT = 'happy green monster with three eyes and horns'  #@param {type:"string"}
SAMPLE_SEED = 42  #@param {type:"integer"}
SAMPLING_STEPS = 64  #@param [16, 32, 64, 128] {type:"raw"}
GUIDANCE = 2.0  #@param {type:"slider", min:0, max:5, step:0.5}
NUMBER_OF_MONSTERS = 8  #@param [1, 4, 8, 16] {type:"raw"}
parsed = parse_prompt(PROMPT)
print('Interpreted as:', caption(parsed))
print('Unseen combination:', tuple(parsed) in HELD_OUT)
generated, denoising_frames = trainer.diffusion.sample(
    trainer.ema, np.repeat(parsed[None], 1 if SMOKE else NUMBER_OF_MONSTERS, axis=0),
    steps=4 if SMOKE else SAMPLING_STEPS, guidance=GUIDANCE, seed=SAMPLE_SEED)
show_grid(generated, heading=caption(parsed), save_path=RUNS/'my_monsters.png')

## 9. Watch reverse diffusion
The first row shows the **actual noisy state** at selected levels, including pure noise at the beginning. It is not a sequence of already-clean predictions. Each frame shows the first monster from the previous batch.

At each step the network predicts noise; the sampler estimates a clean image and moves toward a less noisy state. **DDIM with eta=0** uses no additional random noise after initialization. It is a faster sampler for the same noise-prediction model trained above.

In [ ]:
show_grid(torch.cat([tensor for level, tensor in denoising_frames]),
          [f't={level}' for level, tensor in denoising_frames], columns=5,
          heading='Reverse diffusion — actual noisy states')
gif_path = save_gif(denoising_frames, RUNS/'denoising.gif')
display(DisplayImage(filename=gif_path))

## 10. A fair comparison: change one word, keep the starting noise
If both the prompt and the seed change, we cannot tell what caused the image to change.
Here every monster starts from the **same initial noise**. Only one attribute changes from the first prompt. Does that attribute change? Do other features stay similar?

In [ ]:
comparison_prompts = [
    'happy green monster with two eyes and no horns',
    'happy blue monster with two eyes and no horns',
    'happy green monster with three eyes and no horns',
    'happy green monster with two eyes and horns',
    'sad green monster with two eyes and no horns',
]
comparison_images = []
for prompt in comparison_prompts:
    sample, _ = trainer.diffusion.sample(trainer.ema, [parse_prompt(prompt)],
        steps=4 if SMOKE else 64, guidance=2, seed=2026)
    comparison_images.append(sample)
show_grid(torch.cat(comparison_images), comparison_prompts, columns=5,
          heading='One attribute changes — identical initial noise')

## 11. Can the model combine concepts in a new way?
These **four full combinations are absent from training**. Every individual color, eye count, horn state and expression still occurs in training.

We compare generated samples with actual held-out dataset examples. The reference row shows what the attributes look like; it is not a reconstruction target, and the two rows need not match pixel for pixel.

Record each generated monster's four attributes as correct / incorrect / unclear. Try at least 10 seeds before drawing a conclusion. **One attractive example does not prove generalization.**

In [ ]:
unseen_prompts = [caption(a) for a in HELD_OUT]
unseen_samples, _ = trainer.diffusion.sample(trainer.ema, HELD_OUT,
    steps=4 if SMOKE else 64, guidance=2, seed=314)
show_grid(unseen_samples, unseen_prompts, heading='Model output — unseen combinations')
reference_ids = [next(i for i in np.flatnonzero(splits == 2) if tuple(labels[i]) == attrs)
                 for attrs in HELD_OUT]
show_grid(images[reference_ids], unseen_prompts, heading='Dataset references — excluded from training')

## 12. Optional creative control panel
The form in Section 8 always works, including when widgets are unavailable. This panel lets students change attributes without typing. Generation starts only after clicking the button.

In [ ]:
if not SMOKE:
    import ipywidgets as widgets
    try:
        from google.colab import output as colab_output
        colab_output.enable_custom_widget_manager()
    except ImportError:
        pass
    color_control = widgets.Dropdown(options=COLORS, description='Color:')
    eye_control = widgets.Dropdown(options=EYES, description='Eyes:')
    horn_control = widgets.Checkbox(value=True, description='Horns')
    mood_control = widgets.Dropdown(options=MOODS, description='Mood:')
    seed_control = widgets.BoundedIntText(value=42, min=0, max=2**31-1, description='Seed:')
    guidance_control = widgets.FloatSlider(value=2, min=0, max=5, step=0.5, description='Guidance:')
    generate_button = widgets.Button(description='Generate 4 monsters', button_style='success')
    result_panel = widgets.Output()
    def generate_from_panel(_):
        generate_button.disabled = True
        try:
            with result_panel:
                result_panel.clear_output(wait=True)
                prompt = (f'{mood_control.value} {color_control.value} monster with '
                          f'{eye_control.value} eyes and ' + ('horns' if horn_control.value else 'no horns'))
                print(prompt)
                samples, _ = trainer.diffusion.sample(trainer.ema,
                    np.repeat(parse_prompt(prompt)[None], 4, axis=0), steps=64,
                    guidance=guidance_control.value, seed=seed_control.value)
                show_grid(samples, heading=prompt)
        finally:
            generate_button.disabled = False
    generate_button.on_click(generate_from_panel)
    display(widgets.VBox([widgets.HBox([color_control, eye_control]),
            widgets.HBox([horn_control, mood_control]), seed_control,
            guidance_control, generate_button, result_panel]))

## 13. Save your work and continue later
The training checkpoint is saved every 500 steps and at the end of training. **Colab local storage disappears when its runtime is deleted.** Download a copy or choose the optional Google Drive save below. Nothing is mounted or downloaded automatically.

To resume in a fresh session: run setup, restore `monster.pt` into the printed checkpoint path, and set **RESUME=True** before the configuration/training cells. On the same device type the training random stream is restored too. Cross-device numerical results may differ.

In [ ]:
DOWNLOAD_RESULTS = False  #@param {type:"boolean"}
SAVE_TO_GOOGLE_DRIVE = False  #@param {type:"boolean"}
print('Checkpoint:', CHECKPOINT)
print('Generated image:', RUNS/'my_monsters.png')
print('Denoising animation:', RUNS/'denoising.gif')
if DOWNLOAD_RESULTS and not SMOKE:
    import shutil
    from google.colab import files
    archive_path = shutil.make_archive(str(PROJECT/'monster_results'), 'zip', RUNS)
    files.download(archive_path)
if SAVE_TO_GOOGLE_DRIVE and not SMOKE:
    import shutil
    from google.colab import drive
    drive.mount('/content/drive')
    destination = Path('/content/drive/MyDrive/MonsterGenerator')
    destination.mkdir(parents=True, exist_ok=True)
    shutil.copy2(CHECKPOINT, destination/'monster.pt')
    print('Saved:', destination/'monster.pt')

## Discussion and experiments
1. Why is predicting noise easier to supervise than directly labeling a perfect generated image?
2. Which attributes are easiest to learn: color, horns, number of eyes, or expression? Explain using at least 10 seeds.
3. Compare sampling with 16, 32 and 64 steps using the same seed. What changes in speed and visual quality?
4. Compare guidance 0, 1, 2 and 4. At 0 the prompt is ignored, although a recognizable monster can still appear.
5. Generate the same prompt with 10 seeds. Which features should be fixed by the prompt, and which may vary?
6. Compare familiar and unseen combinations. Keep a table of correct / incorrect / unclear attributes, rather than choosing the best-looking example.
7. Why does this small controlled vocabulary not imply that the model understands arbitrary English?

### If something goes wrong
- **No GPU:** change the runtime to GPU and rerun from Section 1.
- **Out of memory:** set BATCH_SIZE=64 (or 32), RESUME=False, then rerun from Section 5. A resumed checkpoint retains its saved batch size.
- **Mostly noise or malformed eyes:** inspect the preview and loss; set RESUME=True and add training steps. A short time budget is not a quality guarantee.
- **Unknown words:** use the supported attributes and inspect the canonical interpretation printed in Section 8.
- **Checksum mismatch:** reopen the current notebook from GitHub; avoid mixing an old notebook with newer source files.
- **Widget not displaying:** use the Colab form in Section 8.

### Sources and next steps
- Ho et al., [Denoising Diffusion Probabilistic Models](https://arxiv.org/abs/2006.11239).
- Nichol & Dhariwal, [Improved Denoising Diffusion Probabilistic Models](https://arxiv.org/abs/2102.09672): cosine noise schedule.
- Song et al., [Denoising Diffusion Implicit Models](https://arxiv.org/abs/2010.02502): DDIM sampling.
- Ho & Salimans, [Classifier-Free Diffusion Guidance](https://arxiv.org/abs/2207.12598).
- [PyTorch automatic mixed precision](https://docs.pytorch.org/docs/stable/notes/amp_examples.html).

This project intentionally exposes the diffusion equations and a small conditional U-Net. Modern systems extend these ideas with larger datasets, richer text encoders, latent diffusion and more advanced samplers.